In [1]:
import os
print(os.listdir('/content'))

['.config', 'train.zip', 'sample_data']


In [2]:
import zipfile

zip_path = '/content/train.zip'
extract_path = '/content/data'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction completed!")

Extraction completed!


In [3]:
import os
print(os.listdir('/content/data'))

['train']


In [4]:
import os, shutil, random

original_dataset_dir = '/content/data/train'

base_dir = '/content/cats_and_dogs_small'
os.makedirs(base_dir, exist_ok=True)

train_dir = os.path.join(base_dir, 'train')
validation_dir = os.path.join(base_dir, 'validation')
test_dir = os.path.join(base_dir, 'test')

for dir in [train_dir, validation_dir, test_dir]:
    os.makedirs(dir, exist_ok=True)
    os.makedirs(os.path.join(dir, 'cats'), exist_ok=True)
    os.makedirs(os.path.join(dir, 'dogs'), exist_ok=True)

cats = [f for f in os.listdir(original_dataset_dir) if 'cat' in f]
dogs = [f for f in os.listdir(original_dataset_dir) if 'dog' in f]

random.shuffle(cats)
random.shuffle(dogs)

train_cats = cats[:500]
train_dogs = dogs[:500]

val_cats = cats[500:750]
val_dogs = dogs[500:750]

test_cats = cats[750:1000]
test_dogs = dogs[750:1000]

def copy_files(file_list, src_dir, dst_dir):
    for fname in file_list:
        shutil.copyfile(os.path.join(src_dir, fname),
                        os.path.join(dst_dir, fname))

copy_files(train_cats, original_dataset_dir, os.path.join(train_dir, 'cats'))
copy_files(train_dogs, original_dataset_dir, os.path.join(train_dir, 'dogs'))

copy_files(val_cats, original_dataset_dir, os.path.join(validation_dir, 'cats'))
copy_files(val_dogs, original_dataset_dir, os.path.join(validation_dir, 'dogs'))

copy_files(test_cats, original_dataset_dir, os.path.join(test_dir, 'cats'))
copy_files(test_dogs, original_dataset_dir, os.path.join(test_dir, 'dogs'))

print("Sampling completed!")

Sampling completed!


In [5]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_dir = '/content/cats_and_dogs_small/train'
validation_dir = '/content/cats_and_dogs_small/validation'

train_datagen = ImageDataGenerator(rescale=1./255)
validation_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),
    batch_size=20,
    class_mode='binary'
)

validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    target_size=(150, 150),
    batch_size=20,
    class_mode='binary'
)

Found 1000 images belonging to 2 classes.
Found 500 images belonging to 2 classes.


In [6]:
from tensorflow.keras import models, layers

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(150,150,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
history = model.fit(
    train_generator,
    steps_per_epoch=50,
    epochs=10,
    validation_data=validation_generator,
    validation_steps=25
)

Epoch 1/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 65s 1s/step - accuracy: 0.5130 - loss: 0.7186 - val_accuracy: 0.5080 - val_loss: 0.6910
Epoch 2/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 79s 1s/step - accuracy: 0.6240 - loss: 0.6573 - val_accuracy: 0.5680 - val_loss: 0.7324
Epoch 3/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 63s 1s/step - accuracy: 0.6310 - loss: 0.6508 - val_accuracy: 0.6260 - val_loss: 0.6665
Epoch 4/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 64s 1s/step - accuracy: 0.6850 - loss: 0.5746 - val_accuracy: 0.6200 - val_loss: 0.7376
Epoch 5/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 81s 1s/step - accuracy: 0.7620 - loss: 0.4846 - val_accuracy: 0.6180 - val_loss: 0.7297
Epoch 6/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 82s 1s/step - accuracy: 0.8200 - loss: 0.3879 - val_accuracy: 0.6120 - val_loss: 0.8676
Epoch 7/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 63s 1s/step - accuracy: 0.8600 - loss: 0.3156 - val_accuracy: 0.6140 - val_loss: 1.0227
Epoch 8/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 62s 1s/step - accuracy: 0.9220 - loss: 0.1943 - val_accuracy: 0.5940 - val_loss:

In [9]:
train_cats = cats[:1000]
train_dogs = dogs[:1000]

In [10]:
import os, shutil

shutil.rmtree('/content/cats_and_dogs_small')

base_dir = '/content/cats_and_dogs_small'
os.makedirs(base_dir, exist_ok=True)

train_dir = os.path.join(base_dir, 'train')
validation_dir = os.path.join(base_dir, 'validation')
test_dir = os.path.join(base_dir, 'test')

for dir in [train_dir, validation_dir, test_dir]:
    os.makedirs(dir, exist_ok=True)
    os.makedirs(os.path.join(dir, 'cats'), exist_ok=True)
    os.makedirs(os.path.join(dir, 'dogs'), exist_ok=True)

train_cats = cats[:1000]
train_dogs = dogs[:1000]

val_cats = cats[1000:1500]
val_dogs = dogs[1000:1500]

test_cats = cats[1500:2000]
test_dogs = dogs[1500:2000]

def copy_files(file_list, src_dir, dst_dir):
    for fname in file_list:
        shutil.copyfile(os.path.join(src_dir, fname),
                        os.path.join(dst_dir, fname))

original_dataset_dir = '/content/data/train'

copy_files(train_cats, original_dataset_dir, os.path.join(train_dir, 'cats'))
copy_files(train_dogs, original_dataset_dir, os.path.join(train_dir, 'dogs'))

copy_files(val_cats, original_dataset_dir, os.path.join(validation_dir, 'cats'))
copy_files(val_dogs, original_dataset_dir, os.path.join(validation_dir, 'dogs'))

copy_files(test_cats, original_dataset_dir, os.path.join(test_dir, 'cats'))
copy_files(test_dogs, original_dataset_dir, os.path.join(test_dir, 'dogs'))

print("2000 dataset ready")

2000 dataset ready


In [12]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(rescale=1./255)
validation_datagen = ImageDataGenerator(rescale=1./255)

train_dir = '/content/cats_and_dogs_small/train'
validation_dir = '/content/cats_and_dogs_small/validation'

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150,150),
    batch_size=20,
    class_mode='binary'
)

validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    target_size=(150,150),
    batch_size=20,
    class_mode='binary'
)

Found 2000 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.


In [13]:
history = model.fit(
    train_generator,
    steps_per_epoch=100,
    epochs=10,
    validation_data=validation_generator,
    validation_steps=50
)

Epoch 1/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 134s 1s/step - accuracy: 0.7450 - loss: 0.5792 - val_accuracy: 0.6790 - val_loss: 0.6316
Epoch 2/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 124s 1s/step - accuracy: 0.8660 - loss: 0.3158 - val_accuracy: 0.6740 - val_loss: 0.7237
Epoch 3/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 125s 1s/step - accuracy: 0.9520 - loss: 0.1366 - val_accuracy: 0.6750 - val_loss: 1.1049
Epoch 4/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 125s 1s/step - accuracy: 0.9795 - loss: 0.0722 - val_accuracy: 0.6910 - val_loss: 1.1909
Epoch 5/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 124s 1s/step - accuracy: 0.9955 - loss: 0.0189 - val_accuracy: 0.6870 - val_loss: 1.6708
Epoch 6/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 142s 1s/step - accuracy: 0.9985 - loss: 0.0079 - val_accuracy: 0.6940 - val_loss: 1.6692
Epoch 7/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 124s 1s/step - accuracy: 1.0000 - loss: 0.0013 - val_accuracy: 0.6860 - val_loss: 1.9566
Epoch 8/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 132s 1s/step - accuracy: 1.0000 - loss: 4.8684e-04 - val_

In [14]:
train_cats = cats[:1500]
train_dogs = dogs[:1500]

In [15]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras import models, layers

conv_base = VGG16(weights='imagenet',
                  include_top=False,
                  input_shape=(150,150,3))

conv_base.trainable = False

model = models.Sequential([
    conv_base,
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [16]:
history = model.fit(
    train_generator,
    steps_per_epoch=100,
    epochs=10,
    validation_data=validation_generator,
    validation_steps=50
)

Epoch 1/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 721s 7s/step - accuracy: 0.8075 - loss: 0.4489 - val_accuracy: 0.8580 - val_loss: 0.3190
Epoch 2/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 745s 7s/step - accuracy: 0.9120 - loss: 0.2066 - val_accuracy: 0.9000 - val_loss: 0.2441
Epoch 3/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 744s 7s/step - accuracy: 0.9470 - loss: 0.1398 - val_accuracy: 0.8890 - val_loss: 0.2653
Epoch 4/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 682s 7s/step - accuracy: 0.9725 - loss: 0.0871 - val_accuracy: 0.8970 - val_loss: 0.2680
Epoch 5/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 713s 7s/step - accuracy: 0.9830 - loss: 0.0578 - val_accuracy: 0.8950 - val_loss: 0.2775
Epoch 6/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 693s 7s/step - accuracy: 0.9930 - loss: 0.0324 - val_accuracy: 0.8880 - val_loss: 0.3282
Epoch 7/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 678s 7s/step - accuracy: 0.9985 - loss: 0.0181 - val_accuracy: 0.8960 - val_loss: 0.3146
Epoch 8/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 703s 7s/step - accuracy: 1.0000 - loss: 0.0098 - val_accu